# DCC 2026 Mission 2 — 신고자 / 119대원 분류

Google Colab에서 Mission 2의 MFCC + RandomForest 기준선을 학습하고 검증합니다.
각 발화는 JSON의 `startAt`, `endAt` 경계로만 자르며, 텍스트와 발화 순서는 사용하지 않습니다.

**런타임은 CPU로 설정하세요.** librosa와 scikit-learn만 쓰므로 GPU는 필요 없습니다.
(런타임 → 런타임 유형 변경 → 하드웨어 가속기 `CPU`)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# Drive 안의 실제 폴더명에 맞춰 한 번만 수정하세요.
DATA_ROOT = Path('/content/drive/MyDrive/data(inho)')
CODE_DIR = Path('/content/drive/MyDrive/mission2')
MODEL_DIR = Path('/content/drive/MyDrive/saved_models/mission2')
MODEL_PATH = MODEL_DIR / 'mission2_mfcc_rf.joblib'
CACHE_DIR = Path('/content/drive/MyDrive/mission2_cache')
OUTPUT_DIR = Path('/content/drive/MyDrive/outputs')

for folder in (MODEL_DIR, CACHE_DIR, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print('데이터:', DATA_ROOT.exists(), DATA_ROOT)
print('코드  :', CODE_DIR.exists(), CODE_DIR)
print('캐시  :', CACHE_DIR)

## 준비

Drive의 `mission2` 폴더에 저장소 `src/mission2/`의 파일 3개(`common.py`, `train_mfcc_rf.py`,
`inference.py`)를 업로드한 뒤 아래 셀을 실행합니다. 저장소 코드를 고쳤다면 다시 올려야 합니다.

In [ ]:
!pip -q install librosa scikit-learn joblib tqdm soundfile

required_files = ['common.py', 'train_mfcc_rf.py', 'inference.py']
missing = [name for name in required_files if not (CODE_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'CODE_DIR에 업로드되지 않은 파일: {missing}')
print('Mission 2 코드 준비 완료')

## 데이터 확인

출제 규칙상 **Training set은 서울 데이터만** 사용할 수 있습니다. 아래 셀로 실제 폴더 구성과
WAV–JSON 매칭을 확인합니다. Validation은 평가에만 쓰고 학습에 넣지 않습니다.

In [ ]:
# 서울_구급 외에 쓸 수 있는 폴더가 더 있는지 이름만 확인합니다.
# 파일 개수는 다음 셀에서 세므로 여기서는 디렉터리만 훑습니다.
names = set()
for part in sorted(path for path in DATA_ROOT.iterdir() if path.is_dir()):
    for split in ('Training', 'Validation'):
        for kind in ('1.원천데이터', '2.라벨링데이터'):
            folder = part / '대학부 데이터' / split / kind
            if folder.is_dir():
                names.update(f'{split}/{kind}/{sub.name}'
                             for sub in folder.iterdir() if sub.is_dir())

for name in sorted(names):
    print(name)

In [ ]:
def collect_paths(split: str):
    audio_name, label_name = (('TS_서울_구급', 'TL_서울_구급') if split == 'Training'
                              else ('VS_서울_구급', 'VL_서울_구급'))
    wavs, labels = [], []
    for part in sorted(path for path in DATA_ROOT.iterdir() if path.is_dir()):
        base = part / '대학부 데이터' / split
        wavs.extend((base / '1.원천데이터' / audio_name).rglob('*.wav'))
        labels.extend((base / '2.라벨링데이터' / label_name).rglob('*.json'))
    return wavs, labels

# 분할 다운로드 때문에 part별로는 어긋나므로 전체 합계로 확인합니다.
for split in ('Training', 'Validation'):
    wavs, labels = collect_paths(split)
    wav_stems, label_stems = {p.stem for p in wavs}, {p.stem for p in labels}
    print(f'{split}: WAV={len(wavs):,}, JSON={len(labels):,}, '
          f'WAV만={len(wav_stems - label_stems):,}, JSON만={len(label_stems - wav_stems):,}')

## 학습 및 Validation 평가

특징 추출이 대부분의 시간을 차지합니다. 통화 2,000개마다 `CACHE_DIR`에 shard로 저장되므로,
세션이 끊기면 **아래 셀을 그대로 다시 실행**하면 마지막 shard 다음부터 이어집니다.

시작 5분 뒤 진행바의 `s/it`를 확인하세요. **0.35 이하**여야 3~4시간 안에 끝납니다.

In [ ]:
!cd "$CODE_DIR" && python train_mfcc_rf.py \
  --data_root "$DATA_ROOT" \
  --ckpt_path "$MODEL_PATH" \
  --cache_dir "$CACHE_DIR" \
  --n_estimators 500 \
  --max_depth 30 \
  --seed 42

### 결과 읽는 법

- `다수 클래스 기준선` — 무조건 많은 쪽으로만 찍었을 때의 정확도. 이걸 얼마나 넘는지가 실제 성능입니다.
- `Validation Accuracy` — 모델 선택 기준 지표입니다.
- 분류 리포트의 `119대원(0)` / `신고자(1)` F1이 한쪽으로 크게 치우치지 않는지 함께 봅니다.

RandomForest 파라미터만 바꿔 재실험할 때는 같은 `--cache_dir`를 쓰면 특징 추출을 건너뛰고
몇 분 만에 결과가 나옵니다.

## Validation CSV 형식 확인

제출 형식이 맞는지 점검하는 단계입니다. 성능 수치는 학습 셀에서 이미 출력됩니다.

In [ ]:
first_part = next(path for path in sorted(DATA_ROOT.iterdir()) if path.is_dir())
val_base = first_part / '대학부 데이터' / 'Validation'
VAL_WAV_DIR = val_base / '1.원천데이터' / 'VS_서울_구급'
VAL_LABEL_DIR = val_base / '2.라벨링데이터' / 'VL_서울_구급'
VAL_OUTPUT = OUTPUT_DIR / 'mission2_validation_sample.csv'

!cd "$CODE_DIR" && python inference.py \
  --audio_dir "$VAL_WAV_DIR" \
  --label_dir "$VAL_LABEL_DIR" \
  --ckpt_path "$MODEL_PATH" \
  --output "$VAL_OUTPUT"

import pandas as pd
result = pd.read_csv(VAL_OUTPUT)
display(result.head())
print('행 수:', len(result))
assert list(result.columns) == ['audio file name', 'startAt', 'endAt', 'speaker']
assert set(result['speaker'].unique()).issubset({0, 1})

## 최종 제출

주최 측이 제공한 평가 WAV/JSON 폴더로 경로만 교체해 한 번 실행합니다. 평가 데이터에서는
`speaker` 정답을 읽지 않으며, `startAt`, `endAt`으로 구간만 자릅니다.

```bash
python inference.py --audio_dir {wav_folder} --label_dir {json_folder} \
  --ckpt_path {checkpoint_file} --output ./outputs/mission2.csv
```